<a href="https://colab.research.google.com/github/jeehwanhee/AI_learing/blob/main/Kaggle/global_wheat_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
import joblib
from ultralytics import YOLO
import pandas as pd
from tqdm import tqdm
import os
import shutil
from sklearn.model_selection import train_test_split

zip_path1 = '/content/drive/MyDrive/Colab Notebooks/[CV, 객체 탐지] Global Wheat Detection/train.csv.zip'
zip_path2 = '/content/drive/MyDrive/Colab Notebooks/[CV, 객체 탐지] Global Wheat Detection/train.zip'
zip_path3 = '/content/drive/MyDrive/Colab Notebooks/[CV, 객체 탐지] Global Wheat Detection/test.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path1, 'r') as z :
  z.extractall(extract_path)
with zipfile.ZipFile(zip_path2, 'r') as z :
  z.extractall(os.path.join(extract_path, 'train'))
with zipfile.ZipFile(zip_path3, 'r') as z :
  z.extractall(os.path.join(extract_path, 'test'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('/content/dataset/train.csv')
os.makedirs('/content/dataset/train/labels', exist_ok=True)

def convert_labels():
  img_size = 1024

  for img_id in tqdm(df['image_id'].unique()):
    boxes = df[df['image_id'] == img_id]

    label_path = f'/content/dataset/train/labels/{img_id}.txt'

    with open(label_path, 'w') as f:
      for _, row in boxes.iterrows():
        x, y, w, h = eval(row['bbox'])

        xc = (x + w/2) / img_size
        yc = (y + h/2) / img_size
        wn = w / img_size
        hn = h / img_size

        f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

convert_labels()

100%|██████████| 3373/3373 [01:05<00:00, 51.63it/s]


In [ ]:
root_path = '/content/dataset/train'
label_path = '/content/dataset/train/labels'
images = [f for f in os.listdir(root_path) if f.endswith('.jpg')]

train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

os.makedirs('/content/dataset/images/train', exist_ok=True)
os.makedirs('/content/dataset/images/val', exist_ok=True)
os.makedirs('/content/dataset/labels/train', exist_ok=True)
os.makedirs('/content/dataset/labels/val', exist_ok=True)

def move_file(file, name):
  for f in file:
    shutil.move(os.path.join(root_path, f), f'/content/dataset/images/{name}/{f}')
    label_f = f.replace('.jpg', '.txt')
    if os.path.exists(os.path.join(label_path, label_f)):
      shutil.move(os.path.join(label_path, label_f), f'/content/dataset/labels/{name}/{label_f}')

move_file(train_imgs, 'train')
move_file(val_imgs, 'val')


In [ ]:
yaml_content = """
path: /content/dataset
train: images/train
val: images/val
nc: 1
names: ['wheat']
"""

with open('wheat.yaml', 'w') as f:
  f.write(yaml_content)

model = YOLO('yolo11m.pt')

model.train(
  data='wheat.yaml',
  epochs=40,
  imgsz=1024,
  batch=4,
  mosaic=1.0,
  mixup=0.1,
  box=7.5,
  cls=0.5,
  overlap_mask=True,
  device=0,
  patience=10,
  close_mosaic=10,
  label_smoothing=0.1,
)


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=wheat.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=64, nms=False, opset=None, optimize=False, opt

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b80e4325f40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
'''
from ultralytics import YOLO

# 1. 마지막으로 저장된 'last.pt' 파일을 불러옵니다.
model = YOLO('runs/detect/train/weights/last.pt')

# 2. resume=True 옵션을 주면 멈춘 에폭부터 다시 시작합니다.
model.train(resume=True)
'''

"\nfrom ultralytics import YOLO\n\n# 1. 마지막으로 저장된 'last.pt' 파일을 불러옵니다.\nmodel = YOLO('runs/detect/train/weights/last.pt')\n\n# 2. resume=True 옵션을 주면 멈춘 에폭부터 다시 시작합니다.\nmodel.train(resume=True)\n"